# Evaluation

Three sections, reading whatever's already in `data/cache/` (Claude) and
`data/cache_gemini/` (Gemini) -- both populated by `01_pipeline.ipynb`, no
extraction happens in this notebook:

1. **Golden set — Claude** — agreement with the 10 hand-labeled patients. No API calls.
2. **Golden set — Gemini** — the same scoring against the same 10 patients, for Gemini's extraction.
3. **LLM-as-judge — Claude** — a second model scores fidelity vs. the transcript for the golden 10. Needs a key; a few calls.
4. **Extra: model consistency** — Claude vs. Gemini agreement, and a cross-model judge comparison. Beyond what the challenge asked for, kept as a bonus.

## Setup

In [ ]:
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

import json, sys
from pathlib import Path
sys.path.insert(0, 'src')
import evaluate

try:
    from dotenv import load_dotenv; load_dotenv('.env')
except ImportError:
    print('!! pip install -r requirements.txt')

HAS_KEY = bool(os.getenv('ANTHROPIC_API_KEY'))
HAS_GEMINI = bool(os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_API_KEY'))
CACHE = 'data/cache'
GEMINI_CACHE = 'data/cache_gemini'
interviews = {d['patient_id']: d['interview_transcript']
              for d in json.loads(Path('data/interviews.json').read_text())}
model_recs = evaluate.load_model_records(CACHE)
gemini_recs = evaluate.load_model_records(GEMINI_CACHE)
print(f'anthropic key: {HAS_KEY} | gemini key: {HAS_GEMINI} | '
      f'claude records: {len(model_recs)} | gemini records: {len(gemini_recs)}')


## 1 · Golden set — Claude

Field-level agreement between Claude's extraction and the 10 hand-labeled patients
in `src/golden_set.py` (`HAND_LABELS`). Zero API calls.

In [ ]:
_ = evaluate.evaluate(CACHE)


## 2 · Golden set — Gemini

The identical scoring (exact match / Jaccard overlap / soft fields), same 10
patients, against Gemini's extraction instead of Claude's -- directly comparable
line for line with section 1 above. Needs `01_pipeline.ipynb`'s Gemini pass to
have run at least the golden 10.

In [ ]:
if not gemini_recs:
    print('no Gemini records found in data/cache_gemini/ -- run the Gemini pass in 01_pipeline.ipynb first')
else:
    _ = evaluate.evaluate(GEMINI_CACHE)


## 3 · LLM-as-judge — Claude

A second model rates fidelity (1-5) of each of the golden 10's Claude extractions
against its transcript and flags hallucinations / misses. Correct uncertainty is
not penalised. Uses a few calls.

In [ ]:
from golden_set import HAND_LABELS

if not HAS_KEY:
    print('no anthropic key -> skipping LLM-as-judge')
elif not model_recs:
    print('no cached records -> run the extraction pipeline first')
else:
    # The golden 10 -- same patients as the manual judge pass (data/outputs/llm_judge_manual.json).
    # Point src/evaluate.py judge at any/all 50 patient_ids from the terminal for broader coverage.
    golden_ids_judge = [p for p in HAND_LABELS if p in model_recs]
    for pid in golden_ids_judge:
        v = evaluate.judge_record(interviews[pid], model_recs[pid].model_dump_json())
        print(f"{pid}: fidelity={v.get('fidelity')}/5 | "
              f"hallucinations={v.get('hallucinations')} | misses={v.get('misses')}")
        print('    ', v.get('comment', ''))


## Reading the results

- Golden set (both models) tells you where each model matches human judgment, and lets you compare accuracy head to head.
- Judge catches fabrication/omission the golden set's fixed fields might miss.

Write up: what's solid, what's shaky, what you'd fix first.